# Exposing AIoT Data & Device Services through a REST API — Lecture 7

**AIoT Device → MQTT → Data Ingestion → MongoDB** becomes **Application → REST API → AIoT Platform**.

## Why an API?
Applications should not need to know MongoDB structures or MQTT topics. They use business-oriented endpoints such as `GET /devices/MCH-007/telemetry/latest` and `POST /devices/MCH-007/commands`.

In [ ]:
from fastapi import FastAPI
app = FastAPI(title='AIoT Device API')
@app.get('/health')
def health(): return {'status':'healthy'}
print('GET /health defined')

## Connect to MongoDB

In [ ]:
from pymongo import MongoClient
client=MongoClient('mongodb://localhost:27017',serverSelectionTimeoutMS=5000)
client.admin.command('ping')
collection=client['aiot_platform']['telemetry']
print('Connected to MongoDB')

## Query latest telemetry

In [ ]:
latest=collection.find_one({'device_id':'MCH-007'},sort=[('timestamp',-1)])
print(latest)

## Query telemetry history

In [ ]:
records=list(collection.find({'device_id':'MCH-007'}).sort('timestamp',-1).limit(10))
print('Records:',len(records))

## Command path

**Application → REST API → MQTT Broker → AIoT Device**

The completed `api.py` validates START/STOP/STATUS and publishes the MQTT command.

In [ ]:
import json, uuid
payload={"command":"STOP","reason":"Maintenance test","correlation_id":f"CMD-{uuid.uuid4().hex[:8].upper()}"}
print(json.dumps(payload,indent=2))


## Live demo

Start the Lecture 6 ingestion service and Lecture 5 device, then:

```bash
uvicorn api:app --reload
```

Open `http://127.0.0.1:8000/docs`.

Test latest telemetry, history, status, and POST `/devices/MCH-007/commands` with `{"command":"STOP"}` and then `{"command":"START"}`.

## Learner challenge

Add `GET /devices/{device_id}/temperature/latest` returning only device ID, temperature and timestamp.

## Key takeaway

REST is the application-facing interface. MQTT remains the device-facing communication mechanism. MongoDB remains the persistent telemetry store.

**Next: Dockerize the AIoT platform.**

In [ ]:
client.close()